<a href="https://colab.research.google.com/github/nithyabala-dev/ai-training-assignments/blob/main/RAG_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-core langchain-community langchain-google-genai faiss-cpu

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

/tmp/ipykernel_16642/4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
import os
import getpass

os.environ["GEMINI_API_KEY"]=getpass.getpass("Enter your API key")

Enter your API key··········


Part 1 — Data Understanding & Preprocessing

In [ ]:
loader=PyPDFLoader('/content/troubleshooting_runbook.pdf')

In [ ]:
documents=loader.load()
print("Total number of pages", len(documents))

Total number of pages 7


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700,
    chunk_overlap  = 100
)

chunks = text_splitter.split_documents(documents)

Part 2 — Embedding & Vector Database

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [ ]:
embeddings=GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2"
)

In [ ]:
vector_store=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [ ]:
query="What is the escalation procedure for an incident?"
results = vector_store.similarity_search_with_score(query=query, k=3)
for result,score in enumerate(results):
    print(score,result)

(Document(id='3a9ef398-69f7-44af-9b2e-2f832428741d', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-11T03:10:31+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-11-11T03:10:31+00:00', 'subject': '(unspecified)', 'title': 'Contoso Platform Troubleshooting Runbook', 'trapped': '/False', 'source': '/content/troubleshooting_runbook.pdf', 'total_pages': 7, 'page': 1, 'page_label': '2'}, page_content='Next\nbusiness day\nDaily\nOwning team backlog\n6. Escalation procedure\nThe primary on-call engineer owns the incident until a named incident commander takes over. For Sev1\nand Sev2 incidents an incident commander must be assigned within 15 minutes. The commander does not\ndebug; they coordinate, keep the timeline, and manage stakeholder communication. If the primary on-call\nhas not acknowledged an alert within the ack target, the page automatically escalates to the secondary\non-call and then to the team le

Part 3 — Retrieval Pipeline

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":1})

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
rag_prompt = ChatPromptTemplate.from_template(
    """Answer questions based on retrieved context
    Retrieved Context: {context}
    Question:{query}

    Provide: Answer, Supporting Evidence"""
)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm=ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

In [ ]:
chain1 = rag_prompt|llm|StrOutputParser()

In [ ]:
retrieved_docs = retriever.invoke(query)
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
print(len(context_text))


In [ ]:
response = chain1.invoke({"context": context_text, "query": query})
print(response)

**Answer:** 
The primary on-call engineer owns the incident until a named incident commander takes over, which must happen within 15 minutes for Sev1 and Sev2 incidents. If the primary on-call engineer does not acknowledge an alert within the acknowledgment target, the page automatically escalates first to the secondary on-call engineer and then to the team lead.

**Supporting Evidence:** 
* "The primary on-call engineer owns the incident until a named incident commander takes over. For Sev1 and Sev2 incidents an incident commander must be assigned within 15 minutes."
* "If the primary on-call has not acknowledged an alert within the ack target, the page automatically escalates to the secondary on-call and then to the team lead."


Part 4 — Answer Generation (RAG)

In [ ]:
GROUNDED_PROMPT = """You are an expert production support assistant.
Answer the question using ONLY the provided runbook.

Rules:
1. If the runbook does not contain enough information, respond with "I cannot answer this based on provided context."
3. Show citation for key information.

Context:
{context}

Query: {query}

Answer:"""

Part 5 — End-to-End RAG Application